# 05 — Numeric Types and Precision in Python

## Learning objectives

By the end of this notebook, you should be able to:

- Distinguish `int`, `float`, and `complex`.
- Use Python's numeric operators and understand floor division and modulo.
- Explain why some decimal fractions cannot be represented exactly as binary floating-point values.
- Recognize rounding behaviour and the difference between `round()` and `int()`.
- Compare floating-point values using a tolerance when appropriate.
- Use `math.isclose()` for approximate comparisons.
- Know when `decimal.Decimal` or `fractions.Fraction` may be useful.
- Avoid common numeric mistakes in programs.

**How to study:** Predict the output before running each code cell. Change the values, rerun the examples, and write down what you observe.

## 1. Python's main numeric types

Python has three built-in numeric types you will commonly encounter:

- `int`: integers such as `12`, `0`, and `-7`.
- `float`: floating-point approximations such as `3.14` and `-0.5`.
- `complex`: numbers with real and imaginary parts, such as `2 + 3j`.

Python integers can grow to large values, limited mainly by available memory. Floats typically use IEEE 754 binary floating-point representation on modern systems.

In [1]:
integer_value = 42
float_value = 42.0
complex_value = 4 + 2j

for value in [integer_value, float_value, complex_value]:
    print(repr(value), "->", type(value).__name__)

42 -> int
42.0 -> float
(4+2j) -> complex


## 2. Numeric operators

Python supports addition (`+`), subtraction (`-`), multiplication (`*`), division (`/`), floor division (`//`), modulo (`%`), and exponentiation (`**`).

In [2]:
a = 17
b = 5

print("a + b =", a + b)
print("a - b =", a - b)
print("a * b =", a * b)
print("a / b =", a / b)
print("a // b =", a // b)
print("a % b =", a % b)
print("a ** b =", a ** b)

a + b = 22
a - b = 12
a * b = 85
a / b = 3.4
a // b = 3
a % b = 2
a ** b = 1419857


### Floor division is not always truncation

`//` rounds the quotient down toward negative infinity. This differs from truncating toward zero, especially for negative numbers.

In [3]:
examples = [(17, 5), (-17, 5), (17, -5), (-17, -5)]

for a, b in examples:
    print(f"{a} // {b} = {a // b}; {a} % {b} = {a % b}")

17 // 5 = 3; 17 % 5 = 2
-17 // 5 = -4; -17 % 5 = 3
17 // -5 = -4; 17 % -5 = -3
-17 // -5 = 3; -17 % -5 = -2


Python's quotient and remainder follow the relationship:

`a == (a // b) * b + (a % b)`

For a nonzero divisor, the remainder has the same sign as the divisor (or is zero).

In [4]:
for a, b in [(17, 5), (-17, 5), (17, -5), (-17, -5)]:
    quotient = a // b
    remainder = a % b
    print(
        f"{a} = ({quotient}) * ({b}) + ({remainder}) -> "
        f"{a == quotient * b + remainder}"
    )

17 = (3) * (5) + (2) -> True
-17 = (-4) * (5) + (3) -> True
17 = (-4) * (-5) + (-3) -> True
-17 = (3) * (-5) + (-2) -> True


## 3. What is floating-point representation?

Computers store floating-point values in binary, not as an unlimited sequence of decimal digits. Many fractions that look simple in decimal have repeating binary expansions, so the closest representable value is stored.

This is a representation limitation, not a Python bug.

In [5]:
print("0.1 =", repr(0.1))
print("0.2 =", repr(0.2))
print("0.1 + 0.2 =", repr(0.1 + 0.2))
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)

0.1 = 0.1
0.2 = 0.2
0.1 + 0.2 = 0.30000000000000004
0.1 + 0.2 == 0.3 ? False


### See the difference

The difference from `0.3` is very small. Inspect it in scientific notation to make the scale easier to see.

In [6]:
result = 0.1 + 0.2
difference = result - 0.3

print("Difference:", difference)
print("Difference in scientific notation: {:.20e}".format(difference))

Difference: 5.551115123125783e-17
Difference in scientific notation: 5.55111512312578270212e-17


## 4. Comparing floating-point values

Avoid exact equality checks for results that may contain floating-point rounding error. If the context allows approximate comparison, use a tolerance.

A simple pattern is `abs(a - b) < tolerance`. Python's `math.isclose()` provides a more flexible standard helper with relative and absolute tolerances.

In [7]:
a = 0.1 + 0.2
b = 0.3
tolerance = 1e-9

print("Exact equality:", a == b)
print("Within tolerance:", abs(a - b) < tolerance)

Exact equality: False
Within tolerance: True


In [8]:
import math

a = 0.1 + 0.2
b = 0.3

print(math.isclose(a, b))
print(math.isclose(a, b, rel_tol=1e-9, abs_tol=0.0))
print(math.isclose(1000000.0, 1000000.0001, rel_tol=1e-6))

True
True
True


### Why the tolerance matters

There is no single tolerance that is correct for every problem. A scientific calculation, a currency calculation, and a graphics calculation may need different comparison rules. Choose tolerances based on the scale and meaning of the data.

In [9]:
# Illustrative tolerance comparison
expected = 12.0
measured = 12.00001

for tolerance in [1e-3, 1e-5, 1e-7]:
    print(
        f"tolerance={tolerance:g}: "
        f"{abs(expected - measured) <= tolerance}"
    )

tolerance=0.001: True
tolerance=1e-05: True
tolerance=1e-07: False


## 5. Rounding and truncation

`int(value)` truncates a float toward zero. `round(value)` rounds to the nearest integer when no `ndigits` argument is given.

Python's built-in `round()` uses ties-to-even for exact halfway cases: it rounds to the nearest even result. Floating-point representation can also affect whether a value is exactly halfway.

In [10]:
values = [3.2, 3.5, 4.5, 3.8, -3.2, -3.5, -4.5, -3.8]

for value in values:
    print(f"{value:5} | int: {int(value):3} | round: {round(value):3}")

  3.2 | int:   3 | round:   3
  3.5 | int:   3 | round:   4
  4.5 | int:   4 | round:   4
  3.8 | int:   3 | round:   4
 -3.2 | int:  -3 | round:  -3
 -3.5 | int:  -3 | round:  -4
 -4.5 | int:  -4 | round:  -4
 -3.8 | int:  -3 | round:  -4


You can round to a chosen number of decimal places by supplying `ndigits`. This changes the returned value; it does not make binary floating-point representation exact in general.

In [11]:
measurements = [3.1415926, 2.7182818, 1.005]

for value in measurements:
    print(value, "-> 2 decimal places:", round(value, 2))

3.1415926 -> 2 decimal places: 3.14
2.7182818 -> 2 decimal places: 2.72
1.005 -> 2 decimal places: 1.0


## 6. Scientific notation

Scientific notation is a compact way to write very large or very small floating-point values. In Python, `e` means “times ten to the power of.”

In [12]:
values = [1.5e3, 2.4e-3, 6.02e23, 1e-9]

for value in values:
    print(repr(value), "->", f"{value:.6g}")

1500.0 -> 1500
0.0024 -> 0.0024
6.02e+23 -> 6.02e+23
1e-09 -> 1e-09


## 7. Special floating-point values: infinity and NaN

The `math` module provides positive infinity and NaN (“not a number”). NaN is unusual: it is not equal to itself. Use `math.isnan()` to check for it and `math.isinf()` to check for infinity.

In [13]:
import math

positive_infinity = math.inf
not_a_number = math.nan

print("Infinity:", positive_infinity)
print("Is infinite?", math.isinf(positive_infinity))
print("NaN:", not_a_number)
print("NaN == NaN:", not_a_number == not_a_number)
print("Is NaN?", math.isnan(not_a_number))

Infinity: inf
Is infinite? True
NaN: nan
NaN == NaN: False
Is NaN? True


## 8. When `Decimal` is useful

`decimal.Decimal` supports decimal arithmetic with a configurable precision context. It is useful when decimal representation and rounding rules matter, such as some financial calculations.

Construct a `Decimal` from a string when you want the intended decimal digits. Constructing it directly from a float preserves the float's existing binary approximation.

In [14]:
from decimal import Decimal

print("Float calculation:", 0.1 + 0.2)
print("Decimal calculation:", Decimal("0.1") + Decimal("0.2"))
print("Decimal from float:", Decimal(0.1))

Float calculation: 0.30000000000000004
Decimal calculation: 0.3
Decimal from float: 0.1000000000000000055511151231257827021181583404541015625


### Decimal arithmetic and rounding

`Decimal` supports explicit rounding policies. The example below quantizes values to two decimal places using the default context rounding mode.

In [15]:
from decimal import Decimal, ROUND_HALF_UP

amount = Decimal("12.345")
two_places = Decimal("0.01")

print("Default context:", amount.quantize(two_places))
print("Half up:", amount.quantize(two_places, rounding=ROUND_HALF_UP))

Default context: 12.34
Half up: 12.35


## 9. When `Fraction` is useful

`fractions.Fraction` represents rational numbers exactly as a ratio of integers, when the input is rational. It can be useful for fractions such as one-third or exact ratios in mathematical examples.

In [16]:
from fractions import Fraction

one_third = Fraction(1, 3)
one_sixth = Fraction(1, 6)

print("One third:", one_third)
print("One third + one sixth:", one_third + one_sixth)
print("As float:", float(one_third))

One third: 1/3
One third + one sixth: 1/2
As float: 0.3333333333333333


Be aware that `Fraction(0.1)` converts the exact binary floating-point value, not the ideal decimal fraction `1/10`. Use `Fraction(1, 10)` or `Fraction("0.1")` when you intend the rational number one-tenth.

In [17]:
print("Fraction from float:", Fraction(0.1))
print("Fraction from string:", Fraction("0.1"))
print("Explicit ratio:", Fraction(1, 10))

Fraction from float: 3602879701896397/36028797018963968
Fraction from string: 1/10
Explicit ratio: 1/10


## 10. Numeric type conversion and mixed operations

Python often promotes numeric values when combining types. For example, dividing two integers with `/` produces a float. Mixing an integer and float in arithmetic generally produces a float. Complex arithmetic produces complex values when appropriate.

In [18]:
expressions = [
    ("int + int", 2 + 3),
    ("int + float", 2 + 3.5),
    ("int / int", 7 / 2),
    ("int // int", 7 // 2),
    ("int + complex", 2 + (3 + 4j)),
]

for label, value in expressions:
    print(f"{label:16} -> {value!r:12} type={type(value).__name__}")

int + int        -> 5            type=int
int + float      -> 5.5          type=float
int / int        -> 3.5          type=float
int // int       -> 3            type=int
int + complex    -> (5+4j)       type=complex


## 11. Integer precision and large values

Python integers can represent very large whole numbers exactly, limited by available memory. This differs from `float`, which has finite precision.

In [19]:
big_integer = 10 ** 100
print("Number of digits:", len(str(big_integer)))
print("Integer remains exact:", big_integer + 1 - big_integer == 1)

large_float = float(big_integer)
print("Float approximation:", large_float)
print("Float has same exact integer value?", int(large_float) == big_integer)

Number of digits: 101
Integer remains exact: True
Float approximation: 1e+100
Float has same exact integer value? False


## 12. Common numeric pitfalls

- Using `==` for floating-point results that accumulate rounding error.
- Assuming `int()` rounds to the nearest integer.
- Assuming `//` truncates toward zero for negative values.
- Using `float` for exact decimal money calculations without considering rounding requirements.
- Constructing `Decimal` from a float when the intended decimal value is known.
- Choosing a tolerance without considering the scale and meaning of the calculation.

In [20]:
# Compare truncation, floor division, and rounding
negative_value = -3.7

print("int(-3.7):", int(negative_value))
print("floor(-3.7):", math.floor(negative_value))
print("round(-3.7):", round(negative_value))

int(-3.7): -3
floor(-3.7): -4
round(-3.7): -4


## 13. Mini practice exercises

Try these before reading the review cells.

1. Print the type of `7`, `7.0`, and `7 + 0j`.
2. Calculate quotient and remainder for `23` divided by `4`.
3. Show why `0.1 + 0.2 == 0.3` is `False`.
4. Compare two floats using `math.isclose()`.
5. Show the difference between `int(4.9)` and `round(4.9)`.
6. Explain floor division with `-17 // 5`.
7. Add `Decimal("0.1")` and `Decimal("0.2")`.
8. Use `Fraction` to calculate `1/4 + 1/6`.
9. Check whether `math.nan` is NaN and whether `math.inf` is infinite.
10. Explain why the right tolerance depends on the problem.

In [21]:
# Exercises 1–4
for value in [7, 7.0, 7 + 0j]:
    print(repr(value), type(value).__name__)

print("23 // 4 =", 23 // 4)
print("23 % 4 =", 23 % 4)

print("Exact equality:", 0.1 + 0.2 == 0.3)
print("Approximate equality:", math.isclose(0.1 + 0.2, 0.3))

7 int
7.0 float
(7+0j) complex
23 // 4 = 5
23 % 4 = 3
Exact equality: False
Approximate equality: True


In [22]:
# Exercises 5–9
print("int(4.9):", int(4.9))
print("round(4.9):", round(4.9))
print("-17 // 5 =", -17 // 5)

print("Decimal:", Decimal("0.1") + Decimal("0.2"))
print("Fraction:", Fraction(1, 4) + Fraction(1, 6))

print("Is NaN?", math.isnan(math.nan))
print("Is infinite?", math.isinf(math.inf))

int(4.9): 4
round(4.9): 5
-17 // 5 = -4
Decimal: 0.3
Fraction: 5/12
Is NaN? True
Is infinite? True


## 14. Review: worked answers

Use this section to compare your reasoning with example answers. For Exercise 10, the key point is that an appropriate tolerance depends on the unit, scale, measurement uncertainty, and consequences of a false match.

In [23]:
# A practical approximate comparison
def measurements_match(actual, expected, *, tolerance=1e-6):
    return abs(actual - expected) <= tolerance

print(measurements_match(10.0000001, 10.0))
print(measurements_match(10.01, 10.0))
print(measurements_match(10.01, 10.0, tolerance=0.02))

True
False
True


## Completion checklist

- [ ] I understand `int`, `float`, and `complex`.
- [ ] I can use `/`, `//`, `%`, and `**`.
- [ ] I can explain why floating-point values can have small representation errors.
- [ ] I know how to compare floats approximately.
- [ ] I understand the difference between truncation, floor, and rounding.
- [ ] I can recognize `NaN` and infinity.
- [ ] I know when `Decimal` and `Fraction` may be useful.
- [ ] I can explain why tolerances depend on context.

## Key takeaways

- Integers are exact whole numbers; floats have finite binary precision.
- `//` floors the quotient, including for negative numbers.
- Floating-point calculations may not match decimal expectations exactly.
- Use `math.isclose()` or a context-appropriate tolerance for approximate comparisons.
- `int()` truncates toward zero; `round()` follows Python's rounding rules.
- Use `Decimal` for decimal arithmetic with explicit rounding needs and `Fraction` for exact rational arithmetic.
- Pick numeric types and comparison rules based on the meaning of the problem.

## Next notebook

Continue with **06 — Strings and Boolean Values**, where you will study text operations and Boolean logic in more detail.